# DLC: Build Centered Parquets

Load DLC inference outputs, center coordinates on a reference landmark,
and write whole-session parquets.

Outputs go to `pipeline_outputs_v2/dlc_analysis/centered_{mode}/` on MICROSCOPE.
All three centering modes are built by default.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None to process all dates in the
# animal's session range (from configs/animals.yaml).
DATES = None

# Centering modes to build (all three by default)
CENTER_MODES = ["nose", "spout_midpoint", "eye_midpoint"]

# Overwrite mode: "skip" (default) or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.paths import PathResolver
from stroke_orofacial.dlc_io.parquet import build_centered_parquet, _save_dlc_session_params

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [ ]:
resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
dlc_params = defaults.get("dlc_centering", {})
output_root = resolver.pipeline_outputs("dlc_analysis")

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    print(f"\n{'='*60}")
    print(f"Building parquets for {animal_id}: {len(dates)} dates x {len(CENTER_MODES)} modes")
    print(f"{'='*60}")

    n_written, n_skipped, n_error = 0, 0, 0
    for date in dates:
        date_str = date.strftime("%Y%m%d")
        any_written = False
        for mode in CENTER_MODES:
            try:
                result = build_centered_parquet(
                    animal_id, date_str, mode, resolver,
                    params=dlc_params, overwrite=OVERWRITE,
                )
                if result is not None:
                    any_written = True
                    n_written += 1
                else:
                    n_skipped += 1
            except (FileNotFoundError, RuntimeError) as e:
                n_error += 1
                logging.warning("Error %s/%s/%s: %s", animal_id, date_str, mode, e)

        # Per-session params.json
        _save_dlc_session_params(
            output_root, animal_id, date_str, CENTER_MODES,
            dlc_params, skipped=(not any_written),
        )

    print(f"  Written: {n_written}, Skipped: {n_skipped}, Errors: {n_error}")